## Importing các libraries cần thiết
Sử dụng các python libraries BeautifulSoup, json và requests để làm web scraping. Đồng thời import os để lưu dữ liệu về máy và datetime để stamp dữ liệu.

In [1]:
import requests

from bs4 import BeautifulSoup

import os
import json

import time
from datetime import datetime

## Thu thập báo
Bắt đầu từ 1 trang gốc, rồi recursively tìm các trang khác qua tag "a". Đồng thời kiểm tra xem là trang chính hay là trang tin mới thật bằng cách so sánh page_type (page_type = article => trang tin tức).


Sử dụng set chứa các url để đảm bảo không bị nhét trùng vào bộ scraper.

Sử dụng set chứa các article_id đề phòng trường hợp khác url nhưng cùng trang.

In [ ]:
pages = []
marked_urls = set()
marked_articles = set()

def scrape_pages(root, goal):
    if len(pages) == goal:
        return

    marked_urls.add(root)

    wait_time = 0
    response = requests.get(root)
    while response.status_code != 200:
        wait_time += 1
        if wait_time > 10:
            return
        time.sleep(wait_time)
        response = requests.get(root)

    page = BeautifulSoup(response.text, "lxml")

    page_type = page.find("meta", attrs={"name": 'tt_page_type'})
    article_id = page.find("meta", attrs={"name": 'tt_article_id'})
    if page_type != None and article_id != None:
        if page_type["content"] == "article":
            article_id_content = article_id["content"]
            if article_id_content in marked_articles:
                return
            
            marked_articles.add(article_id_content)
            pages.append(page)
            print(root)

    if len(pages) == goal:
        return
    
    for sub_page in page.find_all("a"):
        if not sub_page.has_attr("href"):
            continue

        href = sub_page["href"]
        if not href.startswith("https://vnexpress.net") or "#" in href or href in marked_urls:
            continue

        scrape_pages(href, goal)
scrape_pages("https://vnexpress.net", 500)

# Lưu raw data
Sau khi đã thu thập ít nhất 500 pages, lưu lại trong file json đánh số + timestamp gồm với metadata.

In [ ]:
RAW_DATA_DIR = "../data/raw"
if not os.path.exists(RAW_DATA_DIR):
    os.makedirs(RAW_DATA_DIR)

for page in pages:
    metadata = {
        "url": page.find("meta", attrs={"name": "its_url"})["content"],
        "title": page.find("meta", attrs={"name": "its_title"})["content"],
        "sections": page.find("meta", attrs={"name": "its_subsection"})["content"],
        "tags": page.find("meta", attrs={"name": "its_tag"})["content"],
        "author": page.find("meta", attrs={"name": "its_author"})["content"],
        "word_count": int(page.find("meta", attrs={"name": "its_wordcount"})["content"]),
        "publication": int(page.find("meta", attrs={"name": "its_publication"})["content"]),
        "update_time": int(page.find("meta", attrs={"name": "article_updatetime"})["content"]),
        "description": page.find("meta", attrs={"name": "description"})["content"]
    }
    index = len(os.listdir(RAW_DATA_DIR))
    date = datetime.today().strftime('%Y-%m-%d %H:%M:%S')

    metadata_file = open(f'{RAW_DATA_DIR}/{index}_{date}.json', mode="w", encoding="utf-8")
    metadata_file.write(json.dumps(metadata, indent=4, ensure_ascii=False))
    metadata_file.close()